In [ ]:
# @title Write the run queue and start it in the background (survives this cell)
import subprocess
q = "/content/f3dr/baselines/out/atlas/queue/queue.sh"
open(q, "w").write('#!/bin/bash\n# One change per run against pl_t_s0_ema; finished runs are skipped on relaunch.\nR=/content/f3dr; A=$R/baselines/out/atlas; Q=$A/queue; L=$Q/queue.log\ncd $R\nsay() { echo "$(date +%H:%M) $*" >> $L; }\nscore() { [ -f $A/$1.pt ] || return 0; python $Q/score2.py $R pl_t_s0_ema $1 > $Q/$1.score.log 2>&1 || say "FAILED score $1"; grep -E "^$1 |^RESULT" $Q/$1.score.log >> $L; }\nrun() {\n  name=$1; shift\n  [ -f $Q/$name.done ] && return\n  say "START $name: $*"\n  python baselines/atlas.py train --target mm --no-flip-loss --ema 0.999 "$@" --ckpt $A/$name.pt > $Q/$name.train.log 2>&1\n  rc=$?\n  if [ $rc -eq 0 ]; then say "$(grep -E \'^saved\' $Q/$name.train.log | head -1)"; score $name; score ${name}_ema\n  else say "FAILED $name rc=$rc: $(tail -1 $Q/$name.train.log | cut -c1-160)"; fi\n  echo $rc > $Q/$name.done\n}\nrun n_pos --encoder dinov3l --arch talk --pos-emb\nrun n_pos_s1 --encoder dinov3l --arch talk --pos-emb --seed 1\nrun n_cat --encoder dinov3l --arch talk --cat-drop 0.15\nrun n_rep --encoder dinov3l --arch talk --repulsion 1.0\nsay "QUEUE DONE"\n')
running = subprocess.run(["pgrep", "-fx", f"bash {q}"], capture_output=True, text=True).stdout.strip()
if running:
    print("already running:", running)
else:
    subprocess.Popen(["bash", q], stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    print("started")
